# Data Preprocessing Basics

In this notebook, we will prepare our dataset for Machine Learning:
1. Import libraries
2. Import dataset
3. Handle missing data
4. Encode categorical variables
5. Split into training and test sets
6. Feature scaling


### Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations — used throughout the preprocessing pipeline |
| `pandas` | Loading the CSV and inspecting the dataset |
| `matplotlib` | Optional plots to visualise data distributions |

In [ ]:
# 1. Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Step 2: Load the Dataset

We load `Data.csv` and immediately split it into features (X) and target (y).

**`iloc[:, :-1]`** selects all columns except the last — these are our input features (Country, Age, Salary).
**`iloc[:, -1]`** selects only the last column — the target we want to predict (Purchased: Yes/No).

Notice the printed X already reveals two problems we must fix: the `Country` column contains strings (needs encoding) and there are `nan` values in Age and Salary (needs imputation).

In [ ]:
# 2. Import dataset
dataset = pd.read_csv('../Dataset/Data.csv')

# -----------------------------------------------------------
# Features vs Dependent Variable
#
# In this dataset:
# - Features (independent variables) are:
#       Country, Age, Salary
#   These are the inputs (the information we know).
#
# - Dependent variable (target) is:
#       Purchased
#   This is the output we want to predict.
#
# The goal:
# Given Country, Age and Salary, can we predict whether
# the person purchased (Yes/No)?
# -----------------------------------------------------------

x = dataset.iloc[:, :-1].values  # Features (independent variables)
y = dataset.iloc[:, -1].values   # Dependent variable (target)


In [ ]:
# show the features.
print(x)

In [ ]:
# shpow the dependent variable.
print(y)

### Step 3: Handle Missing Data

Two values are missing: one Age and one Salary. We have several options:

| Strategy | When to use |
|----------|-------------|
| **Delete the row** | Data is abundant and the missing row is a small fraction |
| **Mean imputation** | Feature is roughly symmetric (no outliers skewing the mean) |
| **Median imputation** | Feature has outliers — median is more robust |
| **Mode imputation** | Categorical feature |

We use **mean imputation** here (`strategy='mean'`).

**`imputer.fit(x[:, 1:3])`** — the imputer learns the column means from the data. In a real pipeline, you would fit only on training data and transform both sets with the same means. Fitting on the full dataset here is acceptable for a preprocessing demo, but would be data leakage in a model evaluation context.

We apply the imputer only to columns 1 and 2 (Age, Salary) — not column 0 (Country), which is categorical and handled separately.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')


# Fit the imputer to the features (independent variables)
imputer.fit(x[:, 1:3])  # Only apply to Age and Salary columns
# Transform the features to fill missing values
x[:, 1:3] = imputer.transform(x[:, 1:3])

In [ ]:
print(x)

### Step 4: Encode Categorical Variables

Machine learning algorithms work with numbers. We have two categorical columns:

**Country (feature, 3 categories) → One-Hot Encoding**

One-hot creates a separate binary column for each country. This avoids implying any ordinal relationship between France, Germany, and Spain — they are just three unordered categories.

```
France   → [1, 0, 0]
Germany  → [0, 1, 0]
Spain    → [0, 0, 1]
```

**Purchased (target, 2 categories) → Label Encoding**

The target variable `Yes/No` is binary. `LabelEncoder` converts it to `1/0`. With only 2 values, label encoding is equivalent to one-hot encoding — there is no false ordinal relationship to introduce.

In [ ]:
# 4. Encode categorical variables independent variables
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), [0])  # Apply OneHotEncoder to the first column (Country)
    ],
    remainder='passthrough'  # Keep the other columns as they are
)

x = np.array(ct.fit_transform(x))

In [ ]:
print(x)

In [ ]:
#encode categorical variables dependent variable
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)  # Encode 'Purchased' (Yes/No) as 1/0


In [ ]:
print(y)

### Step 5: Train/Test Split

We hold out 20% of data (2 rows) for testing and use 80% (8 rows) for training.

**Why split the data at all?**

If we evaluated the model on the same data it was trained on, it would report near-perfect performance — it has simply memorised the answers. The test set simulates genuinely new data the model has never seen, giving an unbiased estimate of real-world performance.

**Why split before feature scaling?**

The scaler in the next step will be fit on the training data. If we scaled first, the test set's statistics would influence the scaler — leaking information from the test set into the training process. The split must always come before any data-dependent transformation.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=1
)

In [ ]:
print(x_train)

In [ ]:
print(x_test)

In [ ]:
print(y_train)

In [ ]:
print(y_test)

### Step 6: Feature Scaling

After encoding, our feature matrix has columns with very different ranges:
- One-hot columns: always 0 or 1
- Age: ~27 to 50
- Salary: ~48,000 to 83,000

Many algorithms (SVMs, KNN, gradient descent-based models) are sensitive to these scale differences. A salary difference of 5,000 would completely dominate an age difference of 5 in distance calculations, even if both carry equal predictive information.

**`StandardScaler`** centres each feature to mean=0 and standard deviation=1:
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

**We only scale Age and Salary (`x[:, 3:]`)** — the one-hot encoded columns are already binary (0/1) and scaling them would distort their meaning.

**`fit_transform` on train, `transform` on test** — the scaler learns the mean and std from training data only. The test set is transformed using those same training statistics, ensuring the test set is treated as genuinely unseen data.

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
x_train[:, 3:] = sc.fit_transform(x_train[:, 3:])  # Scale Age and Salary
x_test[:, 3:] = sc.transform(x_test[:, 3:])  # Scale

In [ ]:
print(x_train)

In [ ]:
print(x_test)                                                   